<div align="center">
<h1 style="background: linear-gradient(135deg, #2c3e50 0%, #3498db 100%); color: white; padding: 30px; border-radius: 20px; margin-bottom: 20px;">🔐 KRİPTOQRAFİYA KURSU</h1>
<h2 style="color: #2c3e50; font-size: 2.5em; margin: 20px 0;">Məşğələ 24</h2>
<h2 style="color: #e74c3c; font-size: 2em; margin: 20px 0;">Açar İdarəçiliyi və PKI</h2>
<h3 style="color: #7f8c8d; font-size: 1.5em; margin: 20px 0;">Sertifikatlar, CA-lar və Etibar İnfrastrukturu</h3>
<br>
<div style="background-color: #3498db; color: white; padding: 15px; border-radius: 50px; display: inline-block; font-size: 1.2em;">⏱️ Məşğələ vaxtı: 2 saat</div>
<br><br>
<p><em>Hazırlanma tarixi: 2024</em></p>
</div>

## 📑 Mündəricat

1. [Məşğələnin məqsədləri](#məşğələnin-məqsədləri)
2. [İcra mühiti](#icra-mühiti)
3. [Açarların həyat dövrü](#açarların-həyat-dövrü)
4. [Kök CA, CSR və sertifikat buraxılışı](#kök-ca-csr-və-sertifikat-buraxılışı)
5. [Sertifikat yolunun yoxlanması](#sertifikat-yolunun-yoxlanması)
6. [CRL və OCSP semantikası](#crl-və-ocsp-semantikası)
7. [Certificate Transparency](#certificate-transparency)
8. [HSM və etimad modelləri](#hsm-və-etimad-modelləri)
9. [ACME, kök proqramları və sertifikat profilləri](#acme-kök-proqramları-və-sertifikat-profilləri)
10. [İnteqrasiya edilmiş özünü-yoxlama](#inteqrasiya-edilmiş-özünü-yoxlama)
11. [Ev tapşırığı və mənbələr](#ev-tapşırığı-və-mənbələr)


## 🎯 Məşğələnin məqsədləri

Bu məşğələdən sonra tələbə:

- private key materialının həyat dövrünü və saxlanma sərhədlərini izah edir;
- CSR-in imzasını və tələb olunan adları siyasətə uyğun yoxlayır;
- CA, intermediate CA və TLS server sertifikatını düzgün X.509 genişlənmələri ilə yaradır;
- imzanın kriptoqrafik yoxlanmasını tam sertifikat yolu yoxlamasından ayırır;
- trust anchor, SAN, Basic Constraints, Key Usage, EKU və path length şərtlərini tətbiq edir;
- CRL-in issuer, imza, vaxt və serial şərtlərini yoxlayır;
- OCSP və Certificate Transparency nəticələrinin nəyi sübut etmədiyini izah edir.


## 📚 İcra mühiti

Notebook yalnız `cryptography` kitabxanasından istifadə edir. Şəbəkə sorğusu, paket quraşdırılması, `input()`, işçi qovluğun dəyişdirilməsi və diskə private key yazılması yoxdur. Bütün açarlar və sertifikatlar yalnız proses yaddaşında yaradılan tədris obyektləridir.

> Real sistemdə açar generasiyası, saxlanması, audit, backup, rotasiya və məhv siyasəti təşkilatın risk modelinə və tətbiq olunan normativ tələblərə əsaslanmalıdır.


In [ ]:
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec

if find_spec("cryptography") is None:
    raise RuntimeError("Bu notebook üçün 'cryptography' kitabxanası tələb olunur")

try:
    CRYPTOGRAPHY_VERSION = version("cryptography")
except PackageNotFoundError as exc:
    raise RuntimeError("'cryptography' paylanması tapılmadı") from exc

print(f"Məcburi asılılıq hazırdır: cryptography {CRYPTOGRAPHY_VERSION}")


## 🔧 Hazırlıq

İcra aşağıdakı qaydalarla aparılır:

1. bütün hüceyrələr yuxarıdan aşağıya işlədilir;
2. məcburi API mövcud deyilsə, icra dərhal dayanır;
3. test vaxtı sabitdir, buna görə nəticələr təkrarlana bilir;
4. təhlükəsizliyin yoxlanması həm müsbət, həm də mənfi nümunələrdən ibarətdir.


### 3.1 Python və API müqaviləsi

Laboratoriya `cryptography.x509.verification` vasitəsilə həqiqi sertifikat yolu yoxlaması aparır. Bu API trust store, yoxlama vaxtı, server adı və maksimum intermediate dərinliyini ayrıca qəbul edir. Revocation bu laboratoriyada path verifier-ə avtomatik aid edilmir; CRL ayrıca yoxlanır.


In [ ]:
import hashlib
import sys
from datetime import datetime, timedelta, timezone

import cryptography
from cryptography import x509
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding, rsa
from cryptography.x509.oid import ExtendedKeyUsageOID, NameOID
from cryptography.x509.verification import PolicyBuilder, Store, VerificationError

required_symbols = (
    x509.CertificateBuilder,
    x509.CertificateSigningRequestBuilder,
    x509.CertificateRevocationListBuilder,
    PolicyBuilder,
    Store,
)
assert all(required_symbols)
print(f"Python {sys.version_info.major}.{sys.version_info.minor}; cryptography {cryptography.__version__}")


### 3.2 Deterministik laboratoriya sərhədi

Sertifikatların serial nömrələri və RSA açarları kriptoqrafik təsadüfiliklə yaradılır, lakin yoxlama vaxtı sabit saxlanılır. Notebook serialları, private key-ləri və PEM məzmununu çıxışa vermir. Bu, reproduktiv idarəetmə axınıdır; eyni bitlərdən ibarət açar yaratmaq məqsədi daşımır.


In [ ]:
LAB_NOW = datetime(2026, 7, 21, 12, 0, tzinfo=timezone.utc)
LAB_CONTRACT = {
    "network": False,
    "file_writes": False,
    "interactive_input": False,
    "private_material_output": False,
}

assert LAB_NOW.utcoffset() == timedelta(0)
assert not any(LAB_CONTRACT.values())
print("Laboratoriya müqaviləsi təsdiqləndi: lokal, yaddaşdaxili və qeyri-interaktiv")


## 📖 Açar idarəçiliyi problemi

Kriptoqrafik alqoritmin təhlükəsizliyi private key-in bütün həyat dövrü qorunmadıqda kifayət etmir. Əsas aktivlərə private key, onun backup-ları, giriş səlahiyyətləri, audit qeydləri və bərpa prosedurları daxildir. Sertifikat public key-i ada və istifadəyə bağlaya bilər, lakin private key-in necə qorunduğunu öz-özünə sübut etmir.


### Təhdid və nəzarət sərhədləri

| Risk | Nümunə nəzarət | Qeyd |
|---|---|---|
| Açarın oğurlanması | HSM/keystore, least privilege, audit | Məhsul sertifikatı nəzarətin düzgün konfiqurasiyasını əvəz etmir |
| Açarın itirilməsi | Siyasətə uyğun backup və bərpa sınağı | Bəzi signing key-lər üçün backup qəsdən qadağan edilə bilər |
| İcazəsiz istifadə | İki nəfərlik nəzarət, autentifikasiya, rate limit | Məqsəd və risk profilinə görə seçilir |
| Köhnəlmiş kriptoqrafiya | Alqoritm inventarı və migration planı | Təqvimdən çox təhlükə və uyğunluq siqnalları vacibdir |
| Yanlış sertifikat buraxılışı | CSR siyasəti, audit, CT monitorinqi | CT buraxılışı avtomatik bloklamır |


## 🔄 Açarların həyat dövrü

Tipik mərhələlər: tələbin müəyyənləşdirilməsi, təhlükəsiz generasiya, istifadəyə verilmə, girişin idarə edilməsi, monitorinq, rotasiya və ya rekey, kompromis zamanı cavab, ləğv və təhlükəsiz məhv. Konkret mərhələlər və müddətlər açarın roluna görə fərqlənir.


### 4.1 Həyat dövrünün sınaq edilə bilən xüsusiyyətləri

- **Generasiya:** uyğun parametr və təsadüfilik mənbəyi seçilir.
- **Saxlama:** diskə yazmaq zəruridirsə, şifrləmə yalnız bir nəzarətdir; parolun idarəsi ayrıca həll olunur.
- **İstifadə:** alqoritm, padding, məqsəd və səlahiyyət sərhədləri yoxlanır.
- **Rotasiya/rekey:** kompromis, kriptoperiod, həcm, uyğunluq və əməliyyat riski ilə tetiklenir.
- **Ləğv:** etimad edən tərəflərin istifadə etdiyi status mexanizmi və fail davranışı əvvəlcədən müəyyən edilir.
- **Məhv:** aktiv nüsxələr, backup-lar və törəmə materiallar inventar üzrə əhatə olunur.


In [ ]:
def key_storage_demo():
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    passphrase = b"AS24-local-demo-passphrase"
    encrypted_pkcs8 = private_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.BestAvailableEncryption(passphrase),
    )
    loaded_key = serialization.load_pem_private_key(encrypted_pkcs8, password=passphrase)
    same_public_key = (
        loaded_key.public_key().public_numbers()
        == private_key.public_key().public_numbers()
    )

    wrong_password_rejected = False
    try:
        serialization.load_pem_private_key(encrypted_pkcs8, password=b"wrong-password")
    except ValueError:
        wrong_password_rejected = True

    assert b"BEGIN ENCRYPTED PRIVATE KEY" in encrypted_pkcs8
    assert same_public_key and wrong_password_rejected
    return private_key, {
        "format": "encrypted PKCS#8 in memory",
        "round_trip": same_public_key,
        "wrong_password_rejected": wrong_password_rejected,
    }


ephemeral_key, key_storage_result = key_storage_demo()
print(key_storage_result)


### 4.2 Rotasiya və kriptoperiod

“Bütün açarları hər N gündən bir dəyişmək” universal qayda deyil. Qərar ən azı bunlara əsaslanır: açarın məqsədi, qoruma səviyyəsi, istifadə həcmi, alqoritm və parametr dayanıqlığı, sertifikatın etibarlılıq müddəti, kompromis siqnalları, vendor və root-program siyasəti, normativ tələblər və keçidin əməliyyat riski.

Sertifikatın yenilənməsi həmişə yeni açar demək deyil; **renewal** və **rekey** ayrıca qərarlardır. Kompromis şübhəsində planlı rotasiyanı gözləmək olmaz.


### ✍️ Çalışma 1: Açar həyat dövrü — 1 bal

Bir TLS signing key üçün həyat dövrü cədvəli qurun. Hər mərhələdə owner, aktiv, əsas risk, nəzarət, audit sübutu və incident trigger göstərin.

**Qəbul meyarı:** ən azı generasiya, saxlanma, istifadə, rekey, ləğv və məhv mərhələləri var; renewal ilə rekey qarışdırılmır; rotasiya müddəti risk arqumenti ilə əsaslandırılır.


## 🏛️ Kök CA, CSR və sertifikat buraxılışı

Bu bölmədə yaddaşdaxili tədris PKI-si yaradılır. Kök CA açarını ümumi serverdə fayl kimi saxlamaq real tətbiq üçün nümunə deyil. Məhsul mühitində offline root, səlahiyyət bölgüsü, ceremony, audit və qorunan açar avadanlığı ayrıca layihələndirilir.


### 5.1 Öz-imzalı kök CA

Öz-imza yalnız sertifikatın daxili imzasını yoxlamağa imkan verir. Kök sertifikatına etimad ayrıca trust-store siyasətindən gəlir. `BasicConstraints(ca=True)`, `keyCertSign`, `crlSign`, SKI və AKI sertifikat profilini aydın göstərir.


In [ ]:
def common_name(value):
    return x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, value)])


def create_root_ca(label="AS24 Tədris Kök CA"):
    key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    subject = common_name(label)
    cert = (
        x509.CertificateBuilder()
        .subject_name(subject)
        .issuer_name(subject)
        .public_key(key.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(LAB_NOW - timedelta(days=1))
        .not_valid_after(LAB_NOW + timedelta(days=3650))
        .add_extension(x509.BasicConstraints(ca=True, path_length=1), critical=True)
        .add_extension(
            x509.KeyUsage(
                digital_signature=False,
                content_commitment=False,
                key_encipherment=False,
                data_encipherment=False,
                key_agreement=False,
                key_cert_sign=True,
                crl_sign=True,
                encipher_only=False,
                decipher_only=False,
            ),
            critical=True,
        )
        .add_extension(x509.SubjectKeyIdentifier.from_public_key(key.public_key()), critical=False)
        .add_extension(x509.AuthorityKeyIdentifier.from_issuer_public_key(key.public_key()), critical=False)
        .sign(key, hashes.SHA256())
    )
    return key, cert


root_key, root_cert = create_root_ca()
assert root_cert.subject == root_cert.issuer
assert root_cert.extensions.get_extension_for_class(x509.BasicConstraints).value.ca
print("Kök CA yaradıldı: CA=true, path_length=1, keyCertSign və crlSign aktivdir")


### 5.2 CSR: possession sübutu və tələb olunan adlar

CSR-in imzası müraciət edənin uyğun private key-ə sahib olduğunu göstərir. Bu imza domenə və ya hüquqi şəxsə nəzarəti sübut etmir. CA imzadan sonra subject/SAN, açar parametrləri və tələb olunan profilə öz siyasətini tətbiq etməlidir.


In [ ]:
def create_server_csr(dns_names=("example.test", "www.example.test")):
    key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    csr = (
        x509.CertificateSigningRequestBuilder()
        .subject_name(common_name(dns_names[0]))
        .add_extension(
            x509.SubjectAlternativeName([x509.DNSName(name) for name in dns_names]),
            critical=False,
        )
        .sign(key, hashes.SHA256())
    )
    return key, csr


server_key, server_csr = create_server_csr()
requested_san = server_csr.extensions.get_extension_for_class(x509.SubjectAlternativeName).value
assert server_csr.is_signature_valid
assert requested_san.get_values_for_type(x509.DNSName) == ["example.test", "www.example.test"]
print("CSR imzası və iki tələb olunan SAN adı yoxlanıldı")


### 5.3 CA siyasəti və server sertifikatının buraxılması

Laboratoriya yalnız əvvəlcədən icazə verilmiş `.test` adlarını qəbul edir və CSR imzasını yoxlayır. Sertifikata yalnız server məqsədi üçün SAN, `serverAuth`, uyğun Key Usage, SKI və issuer-ə bağlı AKI əlavə edilir. 90 günlük müddət tədris parametridir, universal norma deyil.


In [ ]:
def issue_server_certificate(issuer_key, issuer_cert, csr, allowed_names):
    if not csr.is_signature_valid:
        raise ValueError("CSR imzası etibarsızdır")

    san = csr.extensions.get_extension_for_class(x509.SubjectAlternativeName).value
    requested = set(san.get_values_for_type(x509.DNSName))
    if not requested or not requested.issubset(set(allowed_names)):
        raise ValueError("CSR-də siyasətə uyğun olmayan DNS adı var")

    cert = (
        x509.CertificateBuilder()
        .subject_name(csr.subject)
        .issuer_name(issuer_cert.subject)
        .public_key(csr.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(LAB_NOW - timedelta(hours=1))
        .not_valid_after(LAB_NOW + timedelta(days=90))
        .add_extension(x509.BasicConstraints(ca=False, path_length=None), critical=True)
        .add_extension(
            x509.KeyUsage(
                digital_signature=True,
                content_commitment=False,
                key_encipherment=True,
                data_encipherment=False,
                key_agreement=False,
                key_cert_sign=False,
                crl_sign=False,
                encipher_only=False,
                decipher_only=False,
            ),
            critical=True,
        )
        .add_extension(x509.ExtendedKeyUsage([ExtendedKeyUsageOID.SERVER_AUTH]), critical=False)
        .add_extension(san, critical=False)
        .add_extension(x509.SubjectKeyIdentifier.from_public_key(csr.public_key()), critical=False)
        .add_extension(
            x509.AuthorityKeyIdentifier.from_issuer_public_key(issuer_key.public_key()),
            critical=False,
        )
        .sign(issuer_key, hashes.SHA256())
    )
    return cert


allowed_dns_names = {"example.test", "www.example.test"}
direct_server_cert = issue_server_certificate(root_key, root_cert, server_csr, allowed_dns_names)
assert direct_server_cert.issuer == root_cert.subject
assert not direct_server_cert.extensions.get_extension_for_class(x509.BasicConstraints).value.ca
print("Server sertifikatı CSR siyasətindən sonra buraxıldı")


### 5.4 İmzanın yoxlanması və tam path validation

Issuer public key-i ilə sertifikat imzasının uğurlu yoxlanması yalnız kriptoqrafik əlaqəni göstərir. Tam yoxlama trust anchor, issuer zənciri, vaxt, Basic Constraints, Key Usage/EKU, DNS adı və digər siyasət məhdudiyyətlərini də tətbiq etməlidir. Revocation ayrıca siyasətdir.


In [ ]:
def verify_rsa_certificate_signature(cert, issuer_cert):
    issuer_cert.public_key().verify(
        cert.signature,
        cert.tbs_certificate_bytes,
        padding.PKCS1v15(),
        cert.signature_hash_algorithm,
    )
    return True


def validate_direct_server_path(cert, trust_anchor, dns_name):
    verifier = (
        PolicyBuilder()
        .store(Store([trust_anchor]))
        .time(LAB_NOW)
        .max_chain_depth(0)
        .build_server_verifier(x509.DNSName(dns_name))
    )
    return verifier.verify(cert, [])


assert verify_rsa_certificate_signature(direct_server_cert, root_cert)
direct_path = validate_direct_server_path(direct_server_cert, root_cert, "example.test")
assert len(direct_path) == 2

wrong_name_rejected = False
try:
    validate_direct_server_path(direct_server_cert, root_cert, "attacker.test")
except VerificationError:
    wrong_name_rejected = True

assert wrong_name_rejected
print("İmza yoxlandı; tam path validation doğru adı qəbul, yanlış adı rədd etdi")


### ✍️ Çalışma 2: CSR siyasəti — 1 bal

`issue_server_certificate` üçün üç mənfi test əlavə edin: icazəsiz SAN, yanlış CSR imzası və CA kimi istifadə tələb edən profil.

**Qəbul meyarı:** hər test konkret gözlənilən exception-u tutur; ümumi `except Exception` istifadə edilmir; sertifikat buraxılmadığı ayrıca assertion ilə təsdiqlənir.


## 🔗 Sertifikat yolunun yoxlanması

Yol belə qurulur: etibar edən tərəfin əvvəlcədən seçdiyi **trust anchor** → intermediate CA → leaf server sertifikatı. Server root-u “etibarlı” elan edə bilməz; trust anchor client siyasətinin girişidir. Intermediate sertifikat serverin təqdim etdiyi untrusted sertifikatlar arasında ola bilər.


### 6.1 Yoxlanan əlaqələr

Path validation yalnız üç imzanı ardıcıl yoxlamaq deyil. Bu laboratoriyada verifier aşağıdakıları birlikdə tətbiq edir:

- seçilmiş trust store və sabit yoxlama vaxtı;
- issuer/subject və imza əlaqələri;
- CA üçün Basic Constraints, Key Usage və path length;
- leaf üçün SAN, serverAuth EKU və Key Usage;
- tələb olunan DNS adı;
- maksimum bir intermediate CA.

CRL/OCSP yoxlaması bu verifier çağırışına daxil deyil və aşağıda ayrıca göstərilir.


In [ ]:
def issue_intermediate_ca(issuer_key, issuer_cert, label="AS24 Intermediate CA"):
    key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    cert = (
        x509.CertificateBuilder()
        .subject_name(common_name(label))
        .issuer_name(issuer_cert.subject)
        .public_key(key.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(LAB_NOW - timedelta(days=1))
        .not_valid_after(LAB_NOW + timedelta(days=365))
        .add_extension(x509.BasicConstraints(ca=True, path_length=0), critical=True)
        .add_extension(
            x509.KeyUsage(
                digital_signature=False,
                content_commitment=False,
                key_encipherment=False,
                data_encipherment=False,
                key_agreement=False,
                key_cert_sign=True,
                crl_sign=True,
                encipher_only=False,
                decipher_only=False,
            ),
            critical=True,
        )
        .add_extension(x509.SubjectKeyIdentifier.from_public_key(key.public_key()), critical=False)
        .add_extension(
            x509.AuthorityKeyIdentifier.from_issuer_public_key(issuer_key.public_key()),
            critical=False,
        )
        .sign(issuer_key, hashes.SHA256())
    )
    return key, cert


def create_certificate_chain():
    chain_root_key, chain_root_cert = create_root_ca("AS24 Chain Root CA")
    intermediate_key, intermediate_cert = issue_intermediate_ca(chain_root_key, chain_root_cert)
    leaf_key, leaf_csr = create_server_csr(("service.example.test",))
    leaf_cert = issue_server_certificate(
        intermediate_key,
        intermediate_cert,
        leaf_csr,
        {"service.example.test"},
    )
    return {
        "root_key": chain_root_key,
        "root_cert": chain_root_cert,
        "intermediate_key": intermediate_key,
        "intermediate_cert": intermediate_cert,
        "leaf_key": leaf_key,
        "leaf_cert": leaf_cert,
    }


pki_chain = create_certificate_chain()
assert pki_chain["leaf_cert"].issuer == pki_chain["intermediate_cert"].subject
print("Root → intermediate → leaf zənciri yaradıldı")


In [ ]:
def build_server_verifier(root, dns_name, max_depth=1):
    return (
        PolicyBuilder()
        .store(Store([root]))
        .time(LAB_NOW)
        .max_chain_depth(max_depth)
        .build_server_verifier(x509.DNSName(dns_name))
    )


chain_verifier = build_server_verifier(pki_chain["root_cert"], "service.example.test")
verified_path = chain_verifier.verify(
    pki_chain["leaf_cert"],
    [pki_chain["intermediate_cert"]],
)
assert len(verified_path) == 3

negative_path_tests = {}

for label, verifier, intermediates in (
    (
        "wrong_dns_name",
        build_server_verifier(pki_chain["root_cert"], "wrong.example.test"),
        [pki_chain["intermediate_cert"]],
    ),
    ("missing_intermediate", chain_verifier, []),
):
    try:
        verifier.verify(pki_chain["leaf_cert"], intermediates)
        negative_path_tests[label] = False
    except VerificationError:
        negative_path_tests[label] = True

foreign_root_key, foreign_root_cert = create_root_ca("Foreign Root CA")
try:
    build_server_verifier(foreign_root_cert, "service.example.test").verify(
        pki_chain["leaf_cert"],
        [pki_chain["intermediate_cert"]],
    )
    negative_path_tests["wrong_trust_anchor"] = False
except VerificationError:
    negative_path_tests["wrong_trust_anchor"] = True

assert all(negative_path_tests.values())
print({"verified_path_length": len(verified_path), **negative_path_tests})


### ✍️ Çalışma 3: Sertifikat yolu — 1 bal

Mövcud zəncir üçün iki yeni mənfi ssenari yaradın: vaxtı keçmiş leaf və `serverAuth` EKU-su olmayan leaf.

**Qəbul meyarı:** hər sertifikat ayrıca builder ilə qurulur, eyni trust anchor və DNS adı ilə verifier tərəfindən rədd edilir, rəddin səbəbi qısa şərhlə izah olunur.


## ❌ CRL və OCSP semantikası

Revocation status sertifikat yolunun ayrıca siyasət hissəsidir. Status mənbəyini oxumaq kifayət deyil: cavabın hansı issuer və ya səlahiyyətli responder-dən gəldiyi, imzası, aid olduğu sertifikat və vaxt pəncərəsi yoxlanmalıdır. Fail-open/fail-closed davranışı tətbiqin risk modelinə görə seçilir.


In [ ]:
def create_crl(issuer_key, issuer_cert, revoked_certs=()):
    builder = (
        x509.CertificateRevocationListBuilder()
        .issuer_name(issuer_cert.subject)
        .last_update(LAB_NOW - timedelta(minutes=5))
        .next_update(LAB_NOW + timedelta(days=7))
    )
    for cert in revoked_certs:
        revoked = (
            x509.RevokedCertificateBuilder()
            .serial_number(cert.serial_number)
            .revocation_date(LAB_NOW - timedelta(minutes=1))
            .add_extension(x509.CRLReason(x509.ReasonFlags.key_compromise), critical=False)
            .build()
        )
        builder = builder.add_revoked_certificate(revoked)
    return builder.sign(private_key=issuer_key, algorithm=hashes.SHA256())


def validate_crl_status(crl, issuer_cert, cert, at_time):
    if crl.issuer != issuer_cert.subject or cert.issuer != issuer_cert.subject:
        raise ValueError("CRL və sertifikat eyni issuer-ə aid deyil")
    usage = issuer_cert.extensions.get_extension_for_class(x509.KeyUsage).value
    if not usage.crl_sign:
        raise ValueError("Issuer sertifikatında crlSign icazəsi yoxdur")

    issuer_cert.public_key().verify(
        crl.signature,
        crl.tbs_certlist_bytes,
        padding.PKCS1v15(),
        crl.signature_hash_algorithm,
    )
    if not (crl.last_update_utc <= at_time <= crl.next_update_utc):
        raise ValueError("CRL yoxlama vaxtında aktual deyil")
    return crl.get_revoked_certificate_by_serial_number(cert.serial_number) is not None


leaf_crl = create_crl(
    pki_chain["intermediate_key"],
    pki_chain["intermediate_cert"],
    [pki_chain["leaf_cert"]],
)
assert validate_crl_status(
    leaf_crl,
    pki_chain["intermediate_cert"],
    pki_chain["leaf_cert"],
    LAB_NOW,
)

crl_negative_tests = {}
for label, issuer, when in (
    ("foreign_issuer", pki_chain["root_cert"], LAB_NOW),
    ("stale_crl", pki_chain["intermediate_cert"], LAB_NOW + timedelta(days=8)),
):
    try:
        validate_crl_status(leaf_crl, issuer, pki_chain["leaf_cert"], when)
        crl_negative_tests[label] = False
    except (ValueError, InvalidSignature):
        crl_negative_tests[label] = True

assert all(crl_negative_tests.values())
print({"leaf_revoked": True, **crl_negative_tests})


### 7.2 OCSP (RFC 6960)

OCSP cavabında sertifikat statusu `good`, `revoked` və ya `unknown` ola bilər. `good` yalnız responder-in həmin serial üçün göstərilən vaxtda ləğv qeydi bilmədiyini bildirir; sertifikatın buraxıldığını, bütün path şərtlərinin keçdiyini və ya serverin kimliyini təkbaşına sübut etmir.

İstehlakçı ən azı request/response CertID uyğunluğunu, responder səlahiyyətini və imzasını, `thisUpdate`/`nextUpdate` vaxtlarını, freshness siyasətini və nonce istifadə edilibsə onun uyğunluğunu yoxlamalıdır. Şəbəkə xətasının nəticəsi təhlükəsizlik siyasətidir, “good” statusu deyil.


### 7.3 OCSP stapling

Stapling zamanı server CA/OCSP responder-dən aldığı imzalı cavabı TLS handshake zamanı təqdim edə bilər. Client yenə cavabın imzasını, aidiyyətini və freshness-ni yoxlayır. Stapling məxfilik və gecikmə baxımından faydalıdır, lakin köhnə, yanlış və ya səlahiyyətsiz cavabı etibarlı etmir.

`must-staple` kimi siyasətlər və cavab çatmadıqda davranış client ekosistemindən və tətbiq profilindən asılıdır.


### 7.4 Mexanizmlərin müqayisəsi

| Mexanizm | Sübut vahidi | Əsas yoxlamalar | Məhdudiyyət |
|---|---|---|---|
| CRL | İmzalı siyahı | issuer, imza, update vaxtları, serial | Siyahı böyük və ya gecikmiş ola bilər |
| OCSP | Konkret CertID üçün imzalı status | responder səlahiyyəti, imza, CertID, freshness | Onlayn əlçatanlıq və privacy siyasəti tələb edir |
| OCSP stapling | Serverin təqdim etdiyi OCSP cavabı | OCSP ilə eyni | Server köhnə cavab verə və ya cavabı verməyə bilər |

Heç biri DNS adı və trust anchor daxil olmaqla tam path validation-u əvəz etmir.


### ✍️ Çalışma 4: Ləğv — 1 bal

Eyni intermediate üçün boş, aktual CRL yaradın və `validate_crl_status` ilə leaf-in ləğv edilmədiyini göstərin. Sonra CRL imzasını başqa CA ilə yoxlamağa çalışın.

**Qəbul meyarı:** aktual boş CRL `False` qaytarır; başqa CA üçün konkret `ValueError` və ya `InvalidSignature` gözlənilir; serial yoxlanmasından əvvəl issuer, imza və vaxt yoxlamalarının sırası qorunur.


## 🔍 Certificate Transparency

Certificate Transparency (RFC 9162) TLS server sertifikatları üçün append-only log, inclusion/consistency sübutları və monitorinq modeli verir. CT yanlış buraxılışı görünən edə bilər, lakin sertifikatı bloklamır, ləğv etmir və domen sahibinə avtomatik xəbər vermir. Ekosistem siyasəti, monitor və auditorlar ayrıca lazımdır.

Aşağıdakı kod yalnız prefiksli Merkle heşləmənin lokal konsept demonstrasiyasıdır; RFC 9162 wire formatını, SCT imzasını, log siyasətini və ya real inclusion proof-u reallaşdırmır.


In [ ]:
def ct_leaf_hash(entry):
    return hashlib.sha256(b"\x00" + entry).digest()


def ct_node_hash(left, right):
    return hashlib.sha256(b"\x01" + left + right).digest()


def verify_toy_inclusion(entry, index, proof, expected_root):
    current = ct_leaf_hash(entry)
    for sibling in proof:
        current = ct_node_hash(current, sibling) if index % 2 == 0 else ct_node_hash(sibling, current)
        index //= 2
    return current == expected_root


ct_entries = [b"precertificate-A", b"precertificate-B", b"precertificate-C", b"precertificate-D"]
ct_leaves = [ct_leaf_hash(entry) for entry in ct_entries]
ct_left = ct_node_hash(ct_leaves[0], ct_leaves[1])
ct_right = ct_node_hash(ct_leaves[2], ct_leaves[3])
ct_root = ct_node_hash(ct_left, ct_right)
ct_proof_for_index_2 = [ct_leaves[3], ct_left]

assert verify_toy_inclusion(ct_entries[2], 2, ct_proof_for_index_2, ct_root)
assert not verify_toy_inclusion(b"changed-entry", 2, ct_proof_for_index_2, ct_root)
print({"toy_entries": len(ct_entries), "inclusion": True, "changed_entry_rejected": True})


## 🛡️ HSM və təhlükəsiz açar saxlama

HSM açarı qorunan sərhəddə generasiya və istifadə edə, ixracı məhdudlaşdıra və audit inteqrasiyası verə bilər. Bu imkanlar məhsulun sertifikatından deyil, konkret konfiqurasiya, səlahiyyət modeli, firmware, backup/cluster siyasəti və əməliyyat prosedurlarından asılıdır.

FIPS 140-3 modul təhlükəsizlik tələbləri və sınaq çərçivəsidir; “FIPS validated” etiketi bütün tətbiqin təhlükəsiz və ya düzgün konfiqurasiya olunduğunu avtomatik sübut etmir.


## 🤝 Web of Trust və iyerarxik PKI

İyerarxik PKI-də etimad edən tərəf trust anchor və sertifikat siyasəti seçir, sonra leaf-ə qədər yolu yoxlayır. Web of Trust modelində imzalayan şəxs public key ilə identity arasındakı binding barədə attestation verir; verifier isə həmin imzalayana nə qədər etibar etdiyini özü qiymətləndirir.

Bir istifadəçinin başqasının açarını imzalaması qarşı tərəfin imzalayana etibar etdiyini göstərmir. Etimad istiqaməti, imzanın mənası və qəbul həddi tətbiq siyasətində ayrıca müəyyən edilir.


## 🌐 Real dünya tətbiqləri

Public Web PKI, müəssisədaxili PKI, code signing və cihaz identifikasiyası eyni X.509 konstruksiyalarından istifadə edə bilər, lakin trust store, profil, status və audit siyasətləri eyni deyil. Public TLS nəticələrini private PKI və ya sənəd imzasına avtomatik köçürmək olmaz.


### 11.1 Let's Encrypt və ACME

ACME (RFC 8555) hesab, order, authorization və challenge axını ilə sertifikat buraxılışını avtomatlaşdırır. Challenge-in uğuru CA siyasətinə görə identifikator üzərində nəzarəti göstərir; bu, hüquqi şəxsiyyətin universal sübutu deyil. Private key qorunması, deployment, renewal monitorinqi və incident response operatorun məsuliyyətində qalır.


### 11.2 Kök sertifikat proqramları

Brauzer və əməliyyat sistemi root proqramları hansı CA-ların trust store-a daxil olmasını və public TLS üçün hansı əlavə siyasətlərin tətbiqini müəyyən edir. Trust store-lar və siyasətlər məhsul, versiya və platformaya görə fərqlənə bilər. Serverin göndərdiyi self-signed root client-in trust anchor-unu dəyişmir.


In [ ]:
def inspect_server_profile(cert):
    san = cert.extensions.get_extension_for_class(x509.SubjectAlternativeName).value
    eku = cert.extensions.get_extension_for_class(x509.ExtendedKeyUsage).value
    usage = cert.extensions.get_extension_for_class(x509.KeyUsage).value
    constraints = cert.extensions.get_extension_for_class(x509.BasicConstraints).value
    return {
        "dns_names": san.get_values_for_type(x509.DNSName),
        "server_auth": ExtendedKeyUsageOID.SERVER_AUTH in eku,
        "digital_signature": usage.digital_signature,
        "key_encipherment": usage.key_encipherment,
        "is_ca": constraints.ca,
    }


leaf_profile = inspect_server_profile(pki_chain["leaf_cert"])
assert leaf_profile == {
    "dns_names": ["service.example.test"],
    "server_auth": True,
    "digital_signature": True,
    "key_encipherment": True,
    "is_ca": False,
}
print(leaf_profile)


### 11.3 DV, OV və EV

DV/OV/EV terminləri CA/browser ekosisteminin yoxlama və policy kateqoriyalarıdır. Hamısında TLS client yenə hostname, sertifikat yolu və tətbiq olunan siyasəti yoxlamalıdır. UI göstəricisi və təşkilat məlumatı istifadəçinin qarşı tərəfi düzgün anladığını, saytın təhlükəsiz olduğunu və ya biznes əməliyyatının qanuni olduğunu avtomatik sübut etmir.


### ✍️ Çalışma 5: Trust boundary — 1 bal

Public Web PKI və daxili cihaz PKI-si üçün trust anchor, identifier, issuance sübutu, revocation mənbəyi və monitorinq owner-i sütunları olan müqayisə cədvəli qurun.

**Qəbul meyarı:** ən azı iki siyasət fərqi göstərilir; “HTTP 200” və ya “imza düzgündür” nəticəsi identity/path validation əvəzi kimi istifadə edilmir.


## 💡 Praktik tövsiyələr

1. Trust anchor-u tətbiq siyasətinin girişi kimi idarə edin.
2. Sertifikat imzasını tam path validation adlandırmayın.
3. SAN/identifier, Basic Constraints, KU, EKU, vaxt və path length-i məqsədə uyğun yoxlayın.
4. Revocation üçün mənbə, freshness və fail davranışını sənədləşdirin.
5. Root və intermediate key-lər üçün səlahiyyət bölgüsü, audit və bərpa proseduru qurun.
6. Private key-i log, notebook output-u və repozitoriyaya çıxarmayın.
7. CT-ni prevention və ya revocation deyil, görünürlük və monitorinq mexanizmi kimi istifadə edin.
8. Renewal, rekey və incident-triggered replacement anlayışlarını ayırın.


## 🖥️ İnteqrasiya edilmiş özünü-yoxlama

Son hüceyrə şəbəkəyə çıxmadan və fayl yaratmadan əvvəlki bütün obyektlərin əsas invariantlarını yoxlayır. Burada “keçdi” nəticəsi yalnız laboratoriyanın elan etdiyi əhatəyə aiddir; OS trust store, real CA, real OCSP və CT log inteqrasiyasını təsdiqləmir.


In [ ]:
def run_pki_lab_self_check():
    checks = {
        "encrypted_key_round_trip": key_storage_result["round_trip"],
        "csr_signature": server_csr.is_signature_valid,
        "direct_path": len(direct_path) == 2,
        "chain_path": len(verified_path) == 3,
        "path_negative_tests": all(negative_path_tests.values()),
        "revoked_leaf_detected": validate_crl_status(
            leaf_crl,
            pki_chain["intermediate_cert"],
            pki_chain["leaf_cert"],
            LAB_NOW,
        ),
        "crl_negative_tests": all(crl_negative_tests.values()),
        "toy_ct_inclusion": verify_toy_inclusion(
            ct_entries[2],
            2,
            ct_proof_for_index_2,
            ct_root,
        ),
        "server_profile": leaf_profile["server_auth"] and not leaf_profile["is_ca"],
    }
    assert all(checks.values())
    return checks


pki_lab_checks = run_pki_lab_self_check()
print({"passed": sum(pki_lab_checks.values()), "total": len(pki_lab_checks)})


## 🏠 Ev tapşırığı — 10 bal

Yaddaşdaxili laboratoriyanı iki profil ilə genişləndirin: TLS server və client authentication.

| Meyar | Bal |
|---|---:|
| Profillər üçün ayrıca SAN/EKU/KU siyasəti | 2 |
| CSR imzası və allow-list yoxlaması | 2 |
| Düzgün root/intermediate/leaf genişlənmələri | 2 |
| Hər profil üçün müsbət path validation | 1 |
| Yanlış ad, yanlış trust anchor və yanlış EKU üçün mənfi testlər | 2 |
| Təhdid modeli və revocation/fail siyasətinin qısa izahı | 1 |

**Məhdudiyyətlər:** şəbəkə, `input()`, private key output-u və repozitoriyada açar faylı yoxdur. Gözlənilən exception-lar konkret tutulur.


## 📌 Yekun

- Trust anchor etimad edən tərəfin siyasət girişidir.
- Sertifikat imzasının doğruluğu tam sertifikat yolu yoxlaması deyil.
- CSR imzası key possession göstərir, identifier ownership-i təkbaşına göstərmir.
- Revocation cavabı issuer/responder, imza, CertID/serial və freshness ilə birlikdə yoxlanır.
- CT yanlış buraxılışı görünən edə bilər; prevention, revocation və monitorinqi əvəz etmir.
- Açar rotasiyası üçün universal müddət yoxdur; renewal, rekey və kompromis cavabı ayrı qərarlardır.


## 📚 Əlavə resurslar

- RFC 5280 — Internet X.509 Public Key Infrastructure Certificate and CRL Profile
- RFC 6960 — X.509 Internet Public Key Infrastructure Online Certificate Status Protocol (OCSP)
- RFC 9162 — Certificate Transparency Version 2.0
- RFC 8555 — Automatic Certificate Management Environment (ACME)
- NIST SP 800-57 Part 1 — Recommendation for Key Management
- NIST SP 800-131A — Transitioning the Use of Cryptographic Algorithms and Key Lengths
- FIPS 140-3 — Security Requirements for Cryptographic Modules
- `cryptography` documentation — X.509 verification, certificate, CSR və CRL API-ləri

> Standartın olması konkret tətbiqin həmin standarta uyğun olduğunu göstərmir; profil, versiya və test sübutu ayrıca yoxlanmalıdır.
